In [34]:
from email.iterators import typed_subpart_iterator

import pandas as pd
from plotly.subplots import make_subplots
import plotly.express as px

import numpy as np


TARGET_PATH='./analysis/data.csv'

In [2]:
df = pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [3]:
df.shape

(2409, 17)

In [4]:
df.head(3)

,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.0
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.0
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.0


In [5]:
df.columns.to_list()

['연도',
 '월',
 '분기',
 '청코드',
 '내외항구분',
 '수출입구분명',
 '시설코드',
 '시설명',
 '부두구분명',
 '아외국구분',
 '적공구분',
 '컨테이너수(10피트)',
 '컨테이너수(20피트)',
 '컨테이너수(40피트)',
 '컨테이너수(기타)',
 '전체개수',
 '전체물동량']

In [6]:
df.info() #

<class 'pandas.DataFrame'>
RangeIndex: 2409 entries, 0 to 2408
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연도           2409 non-null   int64  
 1   월            2409 non-null   int64  
 2   분기           2409 non-null   int64  
 3   청코드          2409 non-null   str    
 4   내외항구분        2409 non-null   str    
 5   수출입구분명       2409 non-null   str    
 6   시설코드         2409 non-null   int64  
 7   시설명          2409 non-null   str    
 8   부두구분명        2409 non-null   str    
 9   아외국구분        2409 non-null   str    
 10  적공구분         2409 non-null   str    
 11  컨테이너수(10피트)  2409 non-null   int64  
 12  컨테이너수(20피트)  2409 non-null   int64  
 13  컨테이너수(40피트)  2409 non-null   int64  
 14  컨테이너수(기타)    2409 non-null   int64  
 15  전체개수         2409 non-null   int64  
 16  전체물동량        2409 non-null   float64
dtypes: float64(1), int64(9), str(7)
memory usage: 320.1 KB


In [7]:
df['연도'].unique()

array([2024])

In [8]:
df['전체물동량'].isna().sum()

np.int64(0)

In [9]:
df.columns.to_list()

['연도',
 '월',
 '분기',
 '청코드',
 '내외항구분',
 '수출입구분명',
 '시설코드',
 '시설명',
 '부두구분명',
 '아외국구분',
 '적공구분',
 '컨테이너수(10피트)',
 '컨테이너수(20피트)',
 '컨테이너수(40피트)',
 '컨테이너수(기타)',
 '전체개수',
 '전체물동량']

In [10]:
for col in ['청코드','내외항구분', '수출입구분명','아외국구분','적공구분','부두구분명']:
    print(col,":", df[col].unique())

청코드 : <StringArray>
['신항', '북항', '감천']
Length: 3, dtype: str
내외항구분 : <StringArray>
['외항']
Length: 1, dtype: str
수출입구분명 : <StringArray>
['수입', '수입환적', '수출환적', '수출']
Length: 4, dtype: str
아외국구분 : <StringArray>
['외국선', '아국선']
Length: 2, dtype: str
적공구분 : <StringArray>
['적컨', '공컨']
Length: 2, dtype: str
부두구분명 : <StringArray>
['일반부두', '컨테이너부두']
Length: 2, dtype: str


In [11]:
df.groupby(['청코드','월']).size().unstack(level=0)
# unstack 은 어떤 층을 열로 펼칠것이냐? 즉, 청코드와 월을 묶어서 몇개인지 세고 감천 북항 신항을 열로 펼치는 것

청코드,감천,북항,신항
월,,,
1,7,87,97
2,9,87,101
3,11,94,98
4,8,77,107
5,13,75,113
6,9,70,110
7,10,79,116
8,8,70,112
9,7,89,114


In [12]:
# 환적 비중 계산
df['환적여부']=df['수출입구분명'].isin(['수입환적','수출환적'])

In [13]:
zone_total = df.groupby('청코드')['전체물동량'].sum()
zone_total

청코드
감천        9014.0
북항     6512510.0
신항    17880496.0
Name: 전체물동량, dtype: float64

In [14]:
zone_transship=df[df['환적여부']].groupby('청코드')['전체물동량'].sum()
zone_transship

청코드
감천        6192.00
북항     2314512.75
신항    11176479.25
Name: 전체물동량, dtype: float64

In [15]:
zone_share = (zone_transship/zone_total*100).round(2)
zone_share #성과

청코드
감천    68.69
북항    35.54
신항    62.51
Name: 전체물동량, dtype: float64

In [16]:
df.columns

Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량', '환적여부'],
      dtype='str')

In [17]:
df.iloc[2405]

연도                2024
월                    7
분기                   3
청코드                 북항
내외항구분               외항
수출입구분명              수출
시설코드                 8
시설명             자성대 부두
부두구분명           컨테이너부두
아외국구분              외국선
적공구분                적컨
컨테이너수(10피트)          0
컨테이너수(20피트)       7719
컨테이너수(40피트)       6438
컨테이너수(기타)            2
전체개수             14159
전체물동량          20599.5
환적여부             False
Name: 2405, dtype: object

In [18]:
teu_estimate = df['컨테이너수(10피트)']*0.5+df['컨테이너수(20피트)']*1+df['컨테이너수(40피트)']*2
teu_estimate # 전체물동량 우리가 구한

0          19.0
1          15.0
2           1.0
3          61.0
4         231.0
         ...   
2404    21637.0
2405    20595.0
2406    22324.0
2407     6555.0
2408    11877.0
Length: 2409, dtype: float64

In [19]:
teu_estimate.corr(df['전체물동량'])

np.float64(0.9999898787140992)

In [20]:
grp = df.groupby(['청코드','환적여부']).agg(물동량=('전체물동량','sum'),개수=('전체개수','sum'))
grp['TEU_FACTOR'] = grp['물동량']/grp['개수']

grp

# 신항은 2에 가깝기에 40피트 컨테이너를 처리하고 있고 북항은 1에 가깝기에 20피트 컨테이너를 처리하고 있음

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778

In [21]:
grp2 = df.groupby(['청코드','환적여부','적공구분'])['전체물동량'].sum().unstack(fill_value=0)
grp2

적공구분              공컨           적컨
청코드 환적여부                         
감천  False     1083.0      1739.00
    True         0.0      6192.00
북항  False  1314340.5   2883656.75
    True     45790.0   2268722.75
신항  False  1983822.0   4720194.75
    True    713056.0  10463423.25

In [22]:
grp2['공컨비율(%)'] = ((grp2['공컨'] / (grp2['공컨'] + grp2['적컨'])) * 100).round(2)
grp2


적공구분              공컨           적컨  공컨비율(%)
청코드 환적여부                                  
감천  False     1083.0      1739.00    38.38
    True         0.0      6192.00     0.00
북항  False  1314340.5   2883656.75    31.31
    True     45790.0   2268722.75     1.98
신항  False  1983822.0   4720194.75    29.59
    True    713056.0  10463423.25     6.38

In [23]:
grp_plot = grp.reset_index()
grp_plot['환적여부'] = grp_plot['환적여부'].map({True:'환적',False: '수출입(일반)'})
grp_plot

,청코드,환적여부,물동량,개수,TEU_FACTOR
0,감천,수출입(일반),2822.00,1516,1.861478
1,감천,환적,6192.00,3210,1.928972
2,북항,수출입(일반),4197997.25,2859718,1.467976
3,북항,환적,2314512.75,1569611,1.474577
4,신항,수출입(일반),6704016.75,4140278,1.619219
5,신항,환적,11176479.25,6464959,1.728778


In [24]:
grp_plot = grp_plot[grp_plot['청코드'] !='감천']
grp_plot

,청코드,환적여부,물동량,개수,TEU_FACTOR
2,북항,수출입(일반),4197997.25,2859718,1.467976
3,북항,환적,2314512.75,1569611,1.474577
4,신항,수출입(일반),6704016.75,4140278,1.619219
5,신항,환적,11176479.25,6464959,1.728778


In [68]:
import sys
print(sys.executable)

C:\work\portproject1\.venv\Scripts\python.exe


In [69]:
import nbformat
print(nbformat.__version__)

5.11.1


In [26]:
fig = px.bar(
	grp_plot,
	x='청코드',
	y='TEU_FACTOR',
	color='환적여부',
	barmode='group',
	title='청코드, 환적여부별 컨테이너당 TEU'
)
fig.show()

In [27]:
grp2_plot = grp2.reset_index()
grp2_plot['환적여부'] = grp2_plot['환적여부'].map({True:'환적',False: '수출입(일반)'})

In [28]:
grp2_plot = grp2_plot[grp2_plot['청코드'] !='감천']
grp2_plot

적공구분,청코드,환적여부,공컨,적컨,공컨비율(%)
2,북항,수출입(일반),1314340.5,2883656.75,31.31
3,북항,환적,45790.0,2268722.75,1.98
4,신항,수출입(일반),1983822.0,4720194.75,29.59
5,신항,환적,713056.0,10463423.25,6.38


In [30]:
fig = px.bar(
    grp2_plot,
    x='청코드',
    y='공컨비율(%)',
    color='환적여부',
    barmode='group',
    title='청코드, 환적 여부별 공컨비율'
)

fig.show()

물동량이 몰리는 달에도 우리가 구한 TEU FACTOR와 같은
처리 효율 지표가 안정적으로 유지된다면 해당 항만은 물동량 변화에 잘 대응하는 여유있는 운영능력을 갖추었다라고 판단할 수 있는 것이고
반대로 물동량이 늘었을 때 효율이 떨어진다면 처리능력에 병목이 있다라고 볼 수 있는 것이다.

In [31]:
sinhang = df[df['청코드'] == '신항']

monthly_sin = sinhang.groupby('월').agg(물동량=('전체물동량', 'sum'), 개수=('전체개수', 'sum'))

monthly_sin = monthly_sin.reset_index()
monthly_sin['TEU_FACTOR'] = monthly_sin['물동량'] / monthly_sin['개수']

monthly_sin

,월,물동량,개수,TEU_FACTOR
0,1,1416542.50,846609,1.673196
1,2,1356151.25,803971,1.686816
2,3,1563893.50,932734,1.676677
3,4,1492720.75,891671,1.674071
4,5,1554553.00,920312,1.689159
5,6,1531476.50,912843,1.677700
6,7,1558413.50,922562,1.689224
7,8,1526833.50,900600,1.695351
8,9,1382264.25,815262,1.695485
9,10,1518211.25,896472,1.693540


In [33]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
fig_dual = make_subplots(
    specs=[[{'secondary_y': True}]]
)

fig_dual.add_trace(
    go.Bar(
        x=monthly_sin['월'],
        y=monthly_sin['물동량'],
        name='물동량(TEU)'
    ),
    secondary_y=False,
)

fig_dual.add_trace(
    go.Scatter(
        x=monthly_sin['월'],
        y=monthly_sin['TEU_FACTOR'],
        mode='lines+markers',
        name='TEU_FACTOR',
    ),
    secondary_y=True,
)
fig_dual.update_yaxes(secondary_y=True, range=[1.5,1.8])

fig_dual.show()

In [39]:
# y = w(가중치)x+b(편향)
slope, intercept = np.polyfit(monthly_sin['물동량'],monthly_sin['TEU_FACTOR'],1)
# 뒤에 붙는 숫자는 1차함수인지 2차 함수인지 결정
fig_scatter = go.Figure()
fig_scatter.add_trace(
    go.Scatter(
        x=monthly_sin['물동량'],
        y=monthly_sin['TEU_FACTOR'],
        mode='markers+text',
        text=monthly_sin['월'],
        name='월별 관측치'
    )
)

x_range = [monthly_sin['물동량'].min(), monthly_sin['물동량'].max()]

fig_scatter.add_trace(
    go.Scatter(x=x_range,y=[slope * x + intercept for x in x_range],mode='lines', name='회귀선')
)
fig_scatter.show()
#예측모델